# 01 - Run models on the test sets
Slow notebook. Loads the six trained models, runs them on the reservoir and non-reservoir test sets, computes rolling metrics and saves them. Analysis happens in notebooks 02-04.

Flow-input models are dropped (dead for the paper).

In [ ]:
import sys
from pathlib import Path
FUNCTIONS_PATH = Path("/home/mokr/Reservoir_Modelling/ML_Functions/")
if str(FUNCTIONS_PATH) not in sys.path:
    sys.path.append(str(FUNCTIONS_PATH))

import pickle, shelve, gc
import numpy as np
import pandas as pd
import joblib
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

from ML_functions import TensorCatchmentDataset
from ML_Models import LSTM_With_Embedding, LSTMModel, HydraModel   # needed so torch.load can unpickle the models
from ML_Training import validate_model_scores, lagged_model_predictions, calculate_rolling_metrics

from analysis_config import MODEL_NAMES, RESULTS_DIR
model_names = MODEL_NAMES

## Test data

In [ ]:
test_non_res_tensor_data = torch.load(
    "/perm/mokr/CamelsGB_V2/no_lat_test_non_res_tensor_data.pt"
)
test_non_res_dataset = TensorCatchmentDataset(test_non_res_tensor_data)

BATCH_SIZE = 1
NUM_WORKERS = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_non_reservoir_dataloader = DataLoader(
    test_non_res_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)  # , pin_memory=True, persistent_workers=True

test_res_tensor_data = torch.load("/perm/mokr/CamelsGB_V2/no_lat_test_res_tensor_data.pt")
test_res_dataset = TensorCatchmentDataset(test_res_tensor_data)


test_reservoir_dataloader = DataLoader(
    test_res_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)  # , pin_memory=True, persistent_workers=True

test_non_res_tensor_metadata = torch.load(
    "/perm/mokr/CamelsGB_V2/test_non_res_tensor_metadata.pt"
)

test_res_tensor_metadata = torch.load(
    "/perm/mokr/CamelsGB_V2/test_res_tensor_metadata.pt"
)

## Models

In [ ]:
## Not using river flow
No_Flow_Reservoir_Embedded_LSTM = torch.load("/home/mokr/Reservoir_Modelling/CamelsGB_Models/Reservoir/2003_No_Lat_Reservoir_LSTM_Numlayers1_LSTMhs256_Embhs64_Embos16_Epochs0_Gamma0.8_0.001_0.4.pt",
    weights_only=False,
    map_location=torch.device("cpu"),
)


No_Flow_NonReservoir_Embedded_LSTM = torch.load(
    "/home/mokr/Reservoir_Modelling/CamelsGB_Models/Natural/2003_No_Lat_Natural_LSTM_FlowFalse_Numlayers2_LSTMhs256_Embhs64_Embos16_Epochs2_Gamma0.95_0.001_0.4.pt",
    weights_only=False,
    map_location=torch.device("cpu"),
)


No_Flow_Full_Embedded_LSTM = torch.load(
    "/home/mokr/Reservoir_Modelling/CamelsGB_Models/Full/2003_No_Lat__Full_LSTM_Numlayers2_LSTMhs128_Embhs128_Embos32_Epochs0_Gamma0.95.pt_0.001_0.1",
    weights_only=False,
    map_location=torch.device("cpu"),
)


full_base_path = "/home/mokr/Reservoir_Modelling/CamelsGB_Models/Hydra/2003_No_Lat_Full_Hydra_LSTM_Numlayers1_LSTMhs256_Embhs128_Embos16_BodyOut128_MainEp3_ResEp3_ResLR0.001_Gamma0.95_0.001_0.001"

No_Flow_Full_Hydra_Body = torch.load(
    f"{full_base_path}_Body.pt", weights_only=False, map_location=torch.device("cpu")
)
No_Flow_Full_Hydra_Head = torch.load(
    f"{full_base_path}_Main_Head.pt", weights_only=False, map_location=torch.device("cpu")
)
No_Flow_Full_Hydra_ResHead = torch.load(
    f"{full_base_path}_Res_Embedding_Head.pt",
    weights_only=False,
    map_location=torch.device("cpu"),
)
No_Flow_Full_Hydra_Main = HydraModel(No_Flow_Full_Hydra_Body, No_Flow_Full_Hydra_Head)

# NOTE: NonRes head comes from the ResLR0.0001 run, the others from ResLR0.001 (see notebook)
nonres_base_path = "/home/mokr/Reservoir_Modelling/CamelsGB_Models/Hydra/2003_No_Lat_Full_Hydra_LSTM_Numlayers1_LSTMhs256_Embhs128_Embos16_BodyOut128_MainEp3_ResEp3_ResLR0.0001_Gamma0.95_0.001_0.0001"

No_Flow_Full_Hydra_NonResHead = torch.load(
    f"{nonres_base_path}_NonRes_Embedding_Head.pt",
    weights_only=False,
    map_location=torch.device("cpu"),
)

In [ ]:
no_flow_models = [
    No_Flow_Reservoir_Embedded_LSTM,
    No_Flow_NonReservoir_Embedded_LSTM,
    No_Flow_Full_Embedded_LSTM,
    No_Flow_Full_Hydra_Main,
    [No_Flow_Full_Hydra_Body, No_Flow_Full_Hydra_ResHead],
    [No_Flow_Full_Hydra_Body, No_Flow_Full_Hydra_NonResHead],
]

## Streamflow scaler

In [ ]:
from sklearn.preprocessing import StandardScaler
temporal_scalers = joblib.load("/home/mokr/Reservoir_Modelling/scaler_hydromet.pkl")
scaler_river = StandardScaler()
# Extract the mean, variance, and scale for the -2nd column
scaler_river.mean_ = temporal_scalers.mean_[[-2]]
scaler_river.var_ = temporal_scalers.var_[[-2]]
scaler_river.scale_ = temporal_scalers.scale_[[-2]]

## Lagged predictions (feeds the rolling hydrograph and the rolling metrics)

In [ ]:
max_batches = 2e2
warmup = 88
TIME_INTERVAL = 30

print("Running reservoir lagged predictions...")
no_flow_reservoir_lagged = lagged_model_predictions(
    model_names, no_flow_models, test_reservoir_dataloader,
    streamflow_scaler=scaler_river, warmup=warmup, device="cpu",
    time_interval=TIME_INTERVAL, max_batches=max_batches, use_flow=False,
)

print("Running non-reservoir lagged predictions...")
no_flow_nonreservoir_lagged = lagged_model_predictions(
    model_names, no_flow_models, test_non_reservoir_dataloader,
    streamflow_scaler=scaler_river, warmup=warmup, device="cpu",
    time_interval=TIME_INTERVAL, max_batches=max_batches, use_flow=False,
)

## Rolling metrics and save
**Check before relying on this cell.** It is inferred from `calculate_rolling_metrics`' docstring; the cell that produced the existing `1505_*` shelves was not in the notebook. It writes to `*_rerun` paths so it cannot overwrite your current results. Confirm `window_size` and the metadata match how the 1505 shelves were made.

In [ ]:
rolling_metrics = {
    "no_flow_reservoir":    calculate_rolling_metrics(no_flow_reservoir_lagged,    test_res_tensor_metadata,     window_size=90),
    "no_flow_nonreservoir": calculate_rolling_metrics(no_flow_nonreservoir_lagged, test_non_res_tensor_metadata, window_size=90),
}
# defaultdicts built with a lambda cannot be pickled by shelve, so convert to plain dicts
rolling_metrics = {k: {m: dict(v) for m, v in d.items()} for k, d in rolling_metrics.items()}

lagged = {"no_flow_reservoir": no_flow_reservoir_lagged, "no_flow_nonreservoir": no_flow_nonreservoir_lagged}

with shelve.open(f"{RESULTS_DIR}/no_lat_rolling_metrics_rerun", flag="n") as db:
    for k, v in rolling_metrics.items():
        db[k] = v
with shelve.open(f"{RESULTS_DIR}/no_lat_lagged_results_rerun", flag="n") as db:
    for k, v in lagged.items():
        db[k] = v

## Optional: validate_model_scores
Nothing in the analysis notebooks reads this output (the old `validation_results` pickle load was the only consumer and that cell was dead). Delete this cell if it is not used for the paper.

In [ ]:
max_batches = 1e2
warmup = 10
no_flow_reservoir_validation_results = validate_model_scores(
    model_names, no_flow_models, test_reservoir_dataloader, warmup,
    streamflow_scaler=scaler_river, device="cpu", time_interval=TIME_INTERVAL,
    max_batches=max_batches, use_flow=False,
)
no_flow_nonreservoir_validation_results = validate_model_scores(
    model_names, no_flow_models, test_non_reservoir_dataloader, warmup,
    streamflow_scaler=scaler_river, device="cpu", time_interval=TIME_INTERVAL,
    max_batches=max_batches, use_flow=False,
)